In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!unzip /content/drive/MyDrive/dataset.zip


Streaming output truncated to the last 5000 lines.
  inflating: dataset/healthy_moringa/Augmented_0_5393.jpeg  
  inflating: dataset/healthy_moringa/Augmented_0_5394.jpeg  
  inflating: dataset/healthy_moringa/Augmented_0_5395.jpeg  
  inflating: dataset/healthy_moringa/Augmented_0_5396.jpeg  
  inflating: dataset/healthy_moringa/Augmented_0_5397.jpeg  
  inflating: dataset/healthy_moringa/Augmented_0_5398.jpeg  
  inflating: dataset/healthy_moringa/Augmented_0_540.jpeg  
  inflating: dataset/healthy_moringa/Augmented_0_5400.jpeg  
  inflating: dataset/healthy_moringa/Augmented_0_5402.jpeg  
  inflating: dataset/healthy_moringa/Augmented_0_5404.jpeg  
  inflating: dataset/healthy_moringa/Augmented_0_5405.jpeg  
  inflating: dataset/healthy_moringa/Augmented_0_5406.jpeg  
  inflating: dataset/healthy_moringa/Augmented_0_5407.jpeg  
  inflating: dataset/healthy_moringa/Augmented_0_5408.jpeg  
  inflating: dataset/healthy_moringa/Augmented_0_541.jpeg  
  inflating: dataset/healthy_moringa

In [ ]:
!ls dataset


diseased_moringa  diseased_papaya  healthy_moringa  healthy_papaya


In [ ]:
!pip install tensorflow pillow opencv-python


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

IMG_SIZE = 160
BATCH_SIZE = 32
EPOCHS = 12

DATASET_PATH = "dataset"

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

train_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    subset="training",
    class_mode="categorical"
)

val_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    subset="validation",
    class_mode="categorical"
)

model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation="relu",
                           input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, (3,3), activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, (3,3), activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(train_data.num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS
)

model.save("leaf_disease_model.h5")
print("✅ Model saved as leaf_disease_model.h5")


Found 12699 images belonging to 4 classes.
Found 3173 images belonging to 4 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 158, 158, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 79, 79, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 77, 77, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 38, 38, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 36, 36, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 18, 18, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 41472)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     5,308,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,402,308 (20.61 MB)

 Trainable params: 5,402,308 (20.61 MB)

 Non-trainable params: 0 (0.00 B)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/12
397/397 ━━━━━━━━━━━━━━━━━━━━ 867s 2s/step - accuracy: 0.6710 - loss: 0.7647 - val_accuracy: 0.7750 - val_loss: 0.5536
Epoch 2/12
397/397 ━━━━━━━━━━━━━━━━━━━━ 878s 2s/step - accuracy: 0.7858 - loss: 0.4999 - val_accuracy: 0.8021 - val_loss: 0.4589
Epoch 3/12
397/397 ━━━━━━━━━━━━━━━━━━━━ 937s 2s/step - accuracy: 0.7971 - loss: 0.4777 - val_accuracy: 0.7806 - val_loss: 0.5646
Epoch 4/12
397/397 ━━━━━━━━━━━━━━━━━━━━ 933s 2s/step - accuracy: 0.8120 - loss: 0.4407 - val_accuracy: 0.8194 - val_loss: 0.4227
Epoch 5/12
397/397 ━━━━━━━━━━━━━━━━━━━━ 864s 2s/step - accuracy: 0.8246 - loss: 0.4185 - val_accuracy: 0.8229 - val_loss: 0.4156
Epoch 6/12
397/397 ━━━━━━━━━━━━━━━━━━━━ 933s 2s/step - accuracy: 0.8315 - loss: 0.4052 - val_accuracy: 0.8339 - val_loss: 0.3887
Epoch 7/12
397/397 ━━━━━━━━━━━━━━━━━━━━ 887s 2s/step - accuracy: 0.8310 - loss: 0.4027 - val_accuracy: 0.8336 - val_loss: 0.3888
Epoch 8/12
397/397 ━━━━━━━━━━━━━━━━━━━━ 830s 2s/step - accuracy: 0.8334 - loss: 0.3850 - val_accu

✅ Model saved as leaf_disease_model.h5


In [ ]:
from google.colab import files
files.download("leaf_disease_model.h5")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>